In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math


DATA GENERATION -> 3 clusters with 2D normal distribution

In [ ]:
rng = np.random.default_rng(42)

# True GNN parameter
true_means = np.array([
    [0 , 0],
    [5 , 5],
    [-4, 5]
])

true_cov = np.array([
    [[1, 0.3] , [0.3, 1]],
    [[1.5, 0], [0, 0.8]],
    [[0.8, -0.2], [-0.2, 1.2]]
])

true_weights = np.array([0.3, 0.4, 0.3])

n = 600
K = 3

# Tirage des variable lattente 
Z = rng.choice(K, size=n, p=true_weights)

# Data generation 
X = np.array([
    rng.multivariate_normal(true_means[k], true_cov[k])
    for k in Z
    ])

plt.scatter(X[:,0], X[:,1], s=10)
plt.axis('equal')
plt.show()

Fonction for E-M algo en numpy

In [ ]:
def initialization(x,K,rng) : 
    """Initialization of the paramater for EM algorithm"""
    n, d = x.shape
    
    # Set the weight at 1/K
    weight = np.ones(K) / K
    # choose random x in the set on n data point
    mean = x[rng.choice(n,K,replace= False)]
    # K Id matrix 
    cov = np.array([
        np.eye(d) for _ in range(K)
    ])
    
    return weight, mean, cov


def gaussian_density(x, mean, cov) : 
    """Compute the probability density function of a multivariate Gaussian distribution."""
    x = np.asarray(x)
    mean = np.asarray(mean)
    cov = np.asarray(cov)
    
    n, d = x.shape
    
    invers_cov = np.linalg.inv(cov)   # (2,2)
    diff = (x - mean)                 # (600,2)
    
    det = np.linalg.det(cov)
    normalization = 1 / ((2 * math.pi)**(d/2) * np.sqrt(det))
    
    quadratic = np.sum((diff @ invers_cov) * diff , axis = 1)
    
    return normalization * np.exp(-0.5 * quadratic)


def e_step(x, weights, mean, cov) : 
    """E-step, compute responsibilities"""
    n, d = x.shape
    K = len(weights)
    
    gamma = np.zeros((n,K))
    for k in range(K) : 
        gamma[:, k] = weights[k] * gaussian_density(x, mean[k], cov[k])
    
    normalization = np.sum(gamma,axis=1, keepdims=True)
    gamma = gamma / normalization
    
    return gamma 


def m_step(x, mean, responsibilities ) : 
    """M-step, update of the parameters"""
    n, d = x.shape  #(n, d) 
    K = responsibilities.shape[1] # responsibilities (n,K)
    
    Normalization = np.sum(responsibilities,axis = 0)   #(K,)

    # update weights
    new_weight = Normalization / n  #(K,)
    
    # update mean
    new_mean = (responsibilities.T @ x) / Normalization[:,None]     #(K,d)
    
    # update covariance
    new_cov = np.zeros((K,d,d))
    for k in range(K) : 
        diff = (x - new_mean[k])    #(n,d)
        weighted_diff = responsibilities[:,k,None] * diff   #(n,d)
        new_cov[k] = (diff.T @ weighted_diff ) / Normalization[k] #(d,d)
    
    return [new_weight, new_mean, new_cov]

def log_likelihood(x,weight,mean,cov) : 
    """Compute the log-likelihood"""
    n, d = x.shape
    K = len(weight)
    
    p_x = np.zeros(n)
    for k in range(K) : 
        p_x += weight[k] * gaussian_density(x,mean[k],cov[k])
    
    log_likelihood = np.sum(np.log(p_x))
    return log_likelihood
    
    
def em_algorithm(x, weight, mean, cov, n_iter = 100) : 
    """Run of the Expectation-Maximization Algo"""
    history = []
    
    for i in range(n_iter) : 
        # E-step
        responsibilities = e_step(x,weights= weight, mean= mean, cov= cov)
        #M-step
        weight, mean, cov = m_step(x,mean=mean,responsibilities= responsibilities)
        #compute log-likelihood
        Ll = log_likelihood(x,weight=weight,mean=mean,cov=cov)
        history.append(Ll)
    
    return weight, mean, cov, history
     

Execution de l'E-M algorithm

In [ ]:
rng = np.random.default_rng(42)
n_iter = 100

weight, mean, cov = initialization(x=X,K=K,rng=rng)
em_weight, em_mean, em_cov, em_history = em_algorithm(x=X,weight=weight,mean=mean,cov=cov,n_iter=n_iter)

print("Estimation of the weight" , em_weight)
print("True weight" , true_weights)
print("Estimation of the mean" , em_mean)
print("True mean" , true_means)
print("Estimation of the cov" , em_cov)
print("True cov" , true_cov)

plt.plot(em_history, color="red")
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()